In [1]:
import torch
import torch.nn as nn

class SequenceMetaTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=4096, meta_dim=24, num_classes=3, nhead=4, num_layers=2):
        
        super().__init__()
        
        # Sequence input
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Metadata input
        self.meta_mlp = nn.Sequential(
            nn.Linear(meta_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim)
        )
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim*2, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, seq_tokens, meta_data):
        # seq_tokens: [batch_size, seq_len]
        # meta_data: [batch_size, meta_dim]
        
        x_seq = self.embedding(seq_tokens).permute(1, 0, 2)  # [seq_len, batch_size, embed_dim]
        x_seq = self.transformer(x_seq) # [seq_len, batch_size, embed_dim]
        x_seq = x_seq.permute(1, 0, 2).mean(dim=1)  # [batch_size, embed_dim]
        
        x_meta = self.meta_mlp(meta_data)  # [batch_size, embed_dim]
        
        x = torch.cat([x_seq, x_meta], dim=1)  # [batch_size, embed_dim * 2]
        out = self.classifier(x)
        print(out)
        print(out.dtype)
        
        return out


In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import gzip
import os
import numpy as np

class SequenceMetaDataset(Dataset):
    def __init__(self, file_paths, metadata_df, k=6, vocab=None, max_seq_len=1024, label_col="diagnosis"):
        
        self.k = k
        self.vocab = vocab or self.build_vocab(file_paths)
        self.file_paths = file_paths
        self.metadata_df = metadata_df.set_index("External ID")
        self.label_col = label_col
        self.samples = [os.path.basename(path).split("_")[0] for path in self.file_paths]
        self.max_seq_len = max_seq_len
    
    def build_vocab(self, file_paths): # Build a vocab of all possible iterations of kmers 
        from itertools import product
        import string
        bases = "ACGT"
        kmers = [''.join(p) for p in product(bases, repeat=self.k)]
        return {kmer: idx for idx, kmer in enumerate(kmers)}
    
    def encode_kmers(self, kmers):
        flat_kmers = [kmer for contig in kmers for kmer in contig] # Flatten
        return [self.vocab.get(kmer, 0) for kmer in flat_kmers]
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample_id = self.samples[idx]
        file_path = [path for path in self.file_paths if sample_id in path.name][0]
            
        contig_kmers = []
        for contig in parser.parse_contigs(file_path):
            kmers = parser.kmerizer(contig, kmer=6)
            contig_kmers.append(kmers)
            
        seq_encoded = self.encode_kmers(contig_kmers)[:self.max_seq_len] 
        
        # Set metadata
        metadata_row = self.metadata_df.loc[sample_id]      
        metadata = torch.tensor(metadata_row.drop(self.label_col).astype(np.float32).values, dtype=torch.float32)
        
        # Set label
        label = torch.tensor(metadata_row[self.label_col], dtype=torch.long)
        
        return torch.tensor(seq_encoded, dtype=torch.long), metadata, label

In [17]:
import time
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

def train_model(model, dataloader, epochs=10, batch_size=32, lr=1e-4, device='cpu'):
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=0)

    start_time = time.time()

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (input_seq, input_meta, target_label) in enumerate(dataloader):
            
            print(f"input_seq shape: {input_seq.shape}")
            print(f"input_meta shape: {input_meta.shape}")
            print(f"target_label shape: {target_label.shape}")

            input_seq = input_seq.to(device)
            input_meta = input_meta.to(device)
            target_label = target_label.to(device) 

            optimizer.zero_grad()

            output_label = model(input_seq, input_meta)
            print("Output label calculated")
            print(target_label.dtype)
            print(output_label.dtype)
            loss = loss_fn(output_label, target_label)
            print("Loss calculated")

            loss.backward()
            print("Backward completed")
            optimizer.step()
            print("Step completed")

            running_loss += loss.item()
            
            print(str(epoch+1))

            if (i + 1) % 10 == 0:
                avg_loss = running_loss / 10
                print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(dataloader)}], Loss: {avg_loss:.4f}")
                running_loss = 0.0

    end_time = time.time()
    training_time = end_time - start_time
    print(f"Training time: {training_time}")
    
    return model

In [13]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

def encode_metadata(metadata_df):
    # Encode categorical variables in metadata and labels into numerical values
    
    # Separate target and ID
    target_cols = ["External ID", "diagnosis"]
    cat_cols = ["Age at diagnosis"]

    # Fit encoder
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoded_meta = encoder.fit_transform(metadata_df[cat_cols])

    # Create new metadata DataFrame
    encoded_df = pd.DataFrame(encoded_meta, columns=encoder.get_feature_names_out(cat_cols))

    # Combine with target columns
    metadata_encoded = pd.concat([metadata_df[target_cols].reset_index(drop=True), encoded_df], axis=1)

    # Encode labels
    labels = metadata_df["diagnosis"]
    label_encoder = LabelEncoder()
    metadata_encoded["diagnosis"] = label_encoder.fit_transform(labels)
    
    return metadata_encoded

In [19]:
import utils.parser as parser
from sklearn.model_selection import train_test_split
from pathlib import Path
import glob

# Read sampled metadata 
metadata_df = pd.read_csv("sampled_metadata.csv")

# Load paths and metadata
#assembly_dir = Path("/pool001/robcli/hmp2/") # Set assembyl data directory
assembly_dir = Path("./hmp2/")
all_fna_paths = list(assembly_dir.glob("*.fna.gz"))

# Split by sample_id
train_ids, test_ids = train_test_split(metadata_df["External ID"], test_size=0.2, stratify=metadata_df["diagnosis"])
train_paths = [path for path in all_fna_paths if any(sample_id in path.name for sample_id in train_ids.values)]
test_paths  = [path for path in all_fna_paths if any(sample_id in path.name for sample_id in test_ids.values)]

# Encode labels and metadata
metadata_df = encode_metadata(metadata_df)

# Compute metadata dimensions
meta_dim = metadata_df.drop(columns=["External ID", "diagnosis"]).shape[1]

# Create datasets
train_dataset = SequenceMetaDataset(train_paths, metadata_df, max_seq_len=2048)
test_dataset = SequenceMetaDataset(test_paths, metadata_df, max_seq_len=2048, vocab=train_dataset.vocab)

# Pad sequences to fixed length
def collate_fn(batch):
    seqs, metas, labels = zip(*batch)
    seqs = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    metas = torch.stack(metas)
    labels = torch.stack(labels)
    return seqs, metas, labels

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# Set device
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print("Device:", device)f
#print("CUDA available:", torch.cuda.is_available())

print(torch.backends.mps.is_available())
device = torch.device('cpu')

# Set model and train
model = SequenceMetaTransformer(vocab_size=len(train_dataset.vocab), embed_dim=256, meta_dim=meta_dim, 
                                num_classes=3, nhead=4, num_layers=2)

model = train_model(model, train_loader, device=device)

# Save model 
#project_folder = "./model_output"
#os.makedirs(project_folder, exist_ok=True) 

True


/opt/anaconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


input_seq shape: torch.Size([16, 2048])
input_meta shape: torch.Size([16, 34])
target_label shape: torch.Size([16])
tensor([[ 0.0083, -0.0837,  0.0022],
        [ 0.0015, -0.0847,  0.0128],
        [ 0.0087, -0.0737,  0.0016],
        [ 0.0026, -0.0806, -0.0018],
        [-0.0030, -0.0954,  0.0250],
        [ 0.0047, -0.0712,  0.0048],
        [-0.0055, -0.0828,  0.0171],
        [ 0.0025, -0.0819,  0.0105],
        [-0.0076, -0.0929,  0.0304],
        [-0.0011, -0.0810,  0.0237],
        [-0.0025, -0.0911,  0.0072],
        [-0.0027, -0.0757,  0.0137],
        [ 0.0091, -0.0855,  0.0086],
        [-0.0005, -0.0892,  0.0220],
        [-0.0027, -0.0785,  0.0177],
        [-0.0089, -0.0820,  0.0089]], grad_fn=<AddmmBackward0>)
torch.float32
Output label calculated
torch.int64
torch.float32
Loss calculated
Backward completed
Step completed
1
input_seq shape: torch.Size([16, 2048])
input_meta shape: torch.Size([16, 34])
target_label shape: torch.Size([16])
tensor([[-0.1876, -0.0157,  0.139